# AgentCert Statistical Methods Demo
## 16 Methods — 9 Hypotheses — SRE-Agent v2.1 Certification Dataset


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
np.random.seed(42)

# =============================================================================
# SRE-Agent v2.1 Certification Dataset
# 90 fault-injection runs: 30 per category (Application, Network, Resource)
# Metrics: time-to-detect (TTD), detection success, mitigation success
# SLA: TTD <= 300s, breach rate <= 5%, detection rate >= 95%
# =============================================================================

# --- Application Fault (30 runs) ---
# Stable, predictable. Most runs detect in ~2-3 min. One outlier.
app_ttd_detected = [
    98, 102, 112, 118, 125, 128, 131, 134, 139, 141,
    143, 145, 148, 152, 155, 158, 161, 167, 172, 178,
    185, 192, 198, 210, 225, 245, 287
]  # 27 detected
app_ttd_censored = [1200, 1200, 1200]  # 3 never detected (timeout)
app_detected, app_total = 27, 30
app_mitigated, app_reasoning = 26, 8.4

# --- Network Fault (30 runs) ---
# Bimodal: some fast, some very slow. 15/30 never detected.
net_ttd_detected = [
    18, 25, 32, 41, 47, 55, 63, 72, 79, 88,
    94, 112, 658, 843, 944
]  # 15 detected
net_ttd_censored = [1200] * 15  # 15 never detected
net_detected, net_total = 15, 30
net_mitigated, net_reasoning = 12, 5.2

# --- Resource Fault (30 runs) ---
# Right-skewed, moderate spread.
res_ttd_detected = [
    138, 152, 165, 175, 189, 198, 215, 228, 242, 248,
    255, 261, 275, 285, 298, 310, 325, 341, 380, 412, 539
]  # 21 detected
res_ttd_censored = [1200] * 9  # 9 never detected
res_detected, res_total = 21, 30
res_mitigated, res_reasoning = 18, 7.1

SLA_TTD = 300     # seconds
SLA_BREACH = 0.05 # max 5% breach rate

print("=== SRE-Agent v2.1 Certification Dataset ===")
print(f"Application: {app_detected}/{app_total} detected, median TTD={np.median(app_ttd_detected):.0f}s")
print(f"Network:     {net_detected}/{net_total} detected, median TTD={np.median(net_ttd_detected):.0f}s")
print(f"Resource:    {res_detected}/{res_total} detected, median TTD={np.median(res_ttd_detected):.0f}s")
print(f"SLA: TTD <= {SLA_TTD}s, breach rate <= {SLA_BREACH:.0%}")


## Method 1: Wilson Confidence Interval (H-01, H-02)
Bounds the true detection rate from binary success/fail outcomes. The lower bound is the safety floor.


In [ ]:
from hypothesis_framework.scripts.statistical_tests.wilson_ci import wilson_ci

print("Detection rate per fault category (Wilson 95% CI):")
print("=" * 65)
for name, s, n in [("Application", app_detected, app_total),
                    ("Network",     net_detected, net_total),
                    ("Resource",    res_detected, res_total)]:
    r = wilson_ci(s, n)
    print(f"  {name:12s}: {r.interpretation}")
    print(f"                Lower bound (safety floor): {r.lower:.3f}")
    print()


## Method 2: Bootstrap BCa Confidence Interval (H-01)
Plausible range for IQM of time-to-detect via 10,000 bootstrap resamples.


In [ ]:
from hypothesis_framework.scripts.statistical_tests.bootstrap_bca import bootstrap_bca_ci
from scipy.stats import trim_mean

iqm_fn = lambda x: trim_mean(x, 0.25)

print("Bootstrap BCa CI on IQM of time_to_detect (10,000 resamples):")
print("=" * 65)
for name, data in [("Application", app_ttd_detected),
                    ("Network",     net_ttd_detected),
                    ("Resource",    res_ttd_detected)]:
    r = bootstrap_bca_ci(data, statistic_fn=iqm_fn, n_resamples=10000, random_state=42)
    print(f"  {name:12s}: {r.interpretation}")
    print()


## Method 3: Interquartile Mean (H-01)
Outlier-robust central tendency — drops top/bottom 25%, averages middle 50%.


In [ ]:
from hypothesis_framework.scripts.statistical_tests.iqm import interquartile_mean

print("IQM of time_to_detect per category:")
print("=" * 55)
for name, data in [("Application", app_ttd_detected),
                    ("Network",     net_ttd_detected),
                    ("Resource",    res_ttd_detected)]:
    r = interquartile_mean(data)
    print(f"  {name:12s}: {r.interpretation}")


## Method 4: Shapiro-Wilk Normality Test (H-03)
Gatekeeper: decides whether to use Welch's ANOVA (normal) or Kruskal-Wallis (not normal).
If ANY group fails, use Kruskal-Wallis for all.


In [ ]:
from hypothesis_framework.scripts.statistical_tests.shapiro_wilk import shapiro_wilk_test

print("Shapiro-Wilk normality test on time_to_detect:")
print("=" * 55)
all_normal = True
for name, data in [("Application", app_ttd_detected),
                    ("Network",     net_ttd_detected),
                    ("Resource",    res_ttd_detected)]:
    r = shapiro_wilk_test(data)
    print(f"  {name:12s}: {r.interpretation}")
    if not r.is_normal:
        all_normal = False

print()
if all_normal:
    print("Decision: All groups normal -> use Welch's ANOVA")
else:
    print("Decision: At least one group NOT normal -> use Kruskal-Wallis")


## Method 5: Kruskal-Wallis H Test (H-03)
Non-parametric test: do any fault categories have significantly different detection times?


In [ ]:
from hypothesis_framework.scripts.statistical_tests.kruskal_wallis import kruskal_wallis_test

print("Kruskal-Wallis H test for time_to_detect across fault categories:")
print("=" * 65)
r = kruskal_wallis_test(app_ttd_detected, net_ttd_detected, res_ttd_detected)
print(f"  {r.interpretation}")
print(f"  Groups: Application (n={len(app_ttd_detected)}), Network (n={len(net_ttd_detected)}), Resource (n={len(res_ttd_detected)})")


## Method 6: Mann-Whitney U Test (H-03)
Pairwise follow-up after Kruskal-Wallis: which specific pairs differ?


In [ ]:
from hypothesis_framework.scripts.statistical_tests.mann_whitney import mann_whitney_test

print("Pairwise Mann-Whitney U tests (post-hoc):")
print("=" * 60)
pairs = [("App vs Network",    app_ttd_detected, net_ttd_detected),
         ("App vs Resource",    app_ttd_detected, res_ttd_detected),
         ("Net vs Resource",    net_ttd_detected, res_ttd_detected)]

for name, a, b in pairs:
    r = mann_whitney_test(a, b)
    print(f"  {name:20s}: {r.interpretation}")


## Method 7: Vargha-Delaney A12 Effect Size (H-03)
How large is the practical difference? A12=0.50 means identical; 0.71+ is large.


In [ ]:
from hypothesis_framework.scripts.statistical_tests.vargha_delaney import vargha_delaney_a12

print("Vargha-Delaney A12 effect sizes for time_to_detect:")
print("=" * 65)
for name, a, b in pairs:
    r = vargha_delaney_a12(a, b)
    print(f"  {name:20s}: {r.interpretation}")


## Method 8: Welch's ANOVA (H-03)
Parametric alternative — more powerful when normality holds. Included for comparison.


In [ ]:
from hypothesis_framework.scripts.statistical_tests.welch_anova import welch_anova

print("Welch's ANOVA for time_to_detect:")
print("=" * 55)
r = welch_anova(app_ttd_detected, net_ttd_detected, res_ttd_detected)
print(f"  {r.interpretation}")
if r.warnings:
    print(f"  Warnings: {r.warnings}")


## Method 9: Chi-Square / Fisher's Exact Test (H-04)
Are detection success rates uniform across fault types, or does the average hide a weak spot?


In [ ]:
from hypothesis_framework.scripts.statistical_tests.chi_square_fisher import chi_square_fisher_test

table = [
    [app_detected, app_total - app_detected],  # [27, 3]
    [net_detected, net_total - net_detected],  # [15, 15]
    [res_detected, res_total - res_detected],  # [21, 9]
]
print("Contingency table (detected / not detected):")
print(f"  Application:  {table[0]}")
print(f"  Network:      {table[1]}")
print(f"  Resource:     {table[2]}")
print()

r = chi_square_fisher_test(table)
print(f"Test used: {r.test_used}")
print(f"Result: {r.interpretation}")


## Method 10: Levene's Test + Coefficient of Variation (H-05)
Is the agent consistently good or unpredictably variable?
CV < 0.15 = stable, 0.15-0.30 = moderate, > 0.30 = unreliable.


In [ ]:
from hypothesis_framework.scripts.statistical_tests.levene_cv import levene_cv_test

print("Levene's test + CV for time_to_detect:")
print("=" * 70)
r = levene_cv_test(
    app_ttd_detected, net_ttd_detected, res_ttd_detected,
    labels=["Application", "Network", "Resource"]
)
print(f"  {r.interpretation}")


## Method 11: Wilcoxon Signed-Rank Test (H-06)
One-sample test: is the median detection time significantly below the 300s SLA?


In [ ]:
from hypothesis_framework.scripts.statistical_tests.wilcoxon_signed_rank import wilcoxon_signed_rank

print(f"Wilcoxon signed-rank test (SLA threshold: {SLA_TTD}s):")
print("=" * 65)
for name, data in [("Application", app_ttd_detected),
                    ("Network",     net_ttd_detected),
                    ("Resource",    res_ttd_detected)]:
    r = wilcoxon_signed_rank(data, threshold=SLA_TTD)
    print(f"  {name:12s}: {r.interpretation}")


## Method 12: Exact Binomial Test (H-07)
Is the SLA breach rate provably below 5%?


In [ ]:
from hypothesis_framework.scripts.statistical_tests.exact_binomial import exact_binomial_test

print(f"SLA breach analysis (TTD > {SLA_TTD}s, target breach rate < {SLA_BREACH:.0%}):")
print("=" * 70)
for name, data, total in [("Application", app_ttd_detected, app_total),
                           ("Network",     net_ttd_detected, net_total),
                           ("Resource",    res_ttd_detected, res_total)]:
    breaches = sum(1 for x in data if x > SLA_TTD)
    r = exact_binomial_test(breaches, total, target_rate=SLA_BREACH)
    print(f"  {name:12s}: {r.interpretation}")


## Method 13: TOST — Two One-Sided Tests (H-06)
Proves detection time is demonstrably *within* [0, 300s] equivalence bounds.


In [ ]:
from hypothesis_framework.scripts.statistical_tests.tost import tost_test

print(f"TOST equivalence test (bounds: [0, {SLA_TTD}s]):")
print("=" * 65)
for name, data in [("Application", app_ttd_detected),
                    ("Network",     net_ttd_detected),
                    ("Resource",    res_ttd_detected)]:
    r = tost_test(data, low=0, high=SLA_TTD)
    print(f"  {name:12s}: {r.interpretation}")


## Method 14: CVaR — Conditional Value-at-Risk (H-08)
How bad are the worst-case detection times? CVaR95 = average of the worst 5%.


In [ ]:
from hypothesis_framework.scripts.statistical_tests.cvar import cvar_analysis

print("Tail risk analysis for time_to_detect:")
print("=" * 70)
for name, data in [("Application", app_ttd_detected),
                    ("Network",     net_ttd_detected),
                    ("Resource",    res_ttd_detected)]:
    r = cvar_analysis(data, quantile=0.95, sla_threshold=SLA_TTD)
    print(f"  {name:12s}: {r.interpretation}")


## Method 15: Kaplan-Meier Survival Estimator (H-06)
S(t) = probability of being undetected at time t. Handles right-censored (timed-out) runs.


In [ ]:
from hypothesis_framework.scripts.statistical_tests.kaplan_meier import kaplan_meier_analysis

print(f"Kaplan-Meier survival at SLA threshold ({SLA_TTD}s):")
print("=" * 70)

datasets = [
    ("Application", app_ttd_detected + app_ttd_censored,
     [True] * len(app_ttd_detected) + [False] * len(app_ttd_censored)),
    ("Network", net_ttd_detected + net_ttd_censored,
     [True] * len(net_ttd_detected) + [False] * len(net_ttd_censored)),
    ("Resource", res_ttd_detected + res_ttd_censored,
     [True] * len(res_ttd_detected) + [False] * len(res_ttd_censored)),
]

for name, times, events in datasets:
    r = kaplan_meier_analysis(times, events, sla_threshold=SLA_TTD)
    print(f"  {name:12s}: {r.interpretation}")


## Method 16: CUSUM / EWMA Control Charts (H-09)
Is detection time stable over successive runs, or drifting?


In [ ]:
from hypothesis_framework.scripts.statistical_tests.cusum_ewma import cusum_ewma

print("CUSUM/EWMA drift detection for time_to_detect:")
print("=" * 70)
for name, data in [("Application", app_ttd_detected),
                    ("Network",     net_ttd_detected),
                    ("Resource",    res_ttd_detected)]:
    r = cusum_ewma(data)
    print(f"  {name:12s}: {r.interpretation}")


---
## Certification Verdict Summary

| Category | Verdict | Key Evidence |
|----------|---------|--------------|
| **Application** | **CERTIFIED** | Wilson LB 74.4% > 65%, CV < 0.50, Wilcoxon PASS, IQM CI tight |
| **Network** | **WITHHELD** | Detection 50%, CV 1.28 extreme, CVaR catastrophic, multiple gate failures |
| **Resource** | **CONDITIONAL** | Detection 70% but Wilson LB borderline, moderate CV, needs more runs |
